#  MyAnimeList (MAL) Adathalmaz Feltáró Elemzése
## Minőség, rajongás vagy puszta hype? – A sikeres animék anatómiája

### A projekt célkitűzése és kutatási kérdése
A közösségi média és a tartalomfogyasztó platformok korában egy-egy alkotás sikerét számos különböző mutatóval mérhetjük. Jelen elemzés központi kérdése, hogy **mást mutat-e egy anime tényleges minőségéről és megítéléséről az átlagos értékelés (`Score`), a kedvencek száma (`Favorites`) és a nézői érdeklődés (`Members`)?** 

A kutatás során nemcsak e három fő dimenzió közötti korrelációkat és a toplisták átfedéseit vizsgáljuk meg, hanem felkutatjuk a feltűnően eltérő eseteket (outliereket) is: a túlértékelt közönségkedvenceket, a rétegközönségnek szóló kritikai remekműveket, valamint a kultikus klasszikusokat. Emellett górcső alá vesszük, hogy a megfigyelt mintázatok milyen összefüggésben állnak az animék **típusával (`Type`)**, **megjelenési korával (`Aired` / `Premiered`)**, **epizódszámával (`Episodes`)** és **forrásanyagával (`Source`)**.

---

### Az adathalmazok bemutatása
Az elemzés két, egymást kiegészítő, nagyméretű MyAnimeList (MAL) adathalmazra épül:

1. **`anime.csv` (Metaadatok és aggregált statisztikák):**
   * **Méret:** 17 562 egyedi anime részletes adatlapja.
   * **Tartalom:** Általános információk (cím, típus, epizódszám, stúdió, forrásanyag, műfajok, korhatár-besorolás), közösségi aktivitási mutatók (`Members`, `Favorites`, `Watching`, `Completed`, `Dropped`, `Plan to Watch`), valamint a pontszámok részletes eloszlása (`Score`, illetve a `Score-1`-től `Score-10`-ig terjedő szavazatszámok).
2. **`animelist.csv` (Felhasználói interakciók – Big Data):**
   * **Méret:** ~109 millió sor, amely 325 772 egyedi felhasználó értékeléseit és naplózott tevékenységeit tartalmazza.
   * **Tartalom:** Felhasználói azonosító (`user_id`), anime azonosító (`anime_id`), adott pontszám (`score`), megtekintési státusz (`watching_status`) és a megnézett epizódok száma (`watched_episodes`).

---

### Az elemzés felépítése (Notebook Struktúra)
A Jupyter Notebook a későbbi prezentációs logikát követve 5 fő szakaszra bomlik:

1. **Adatelőkészítés és Technikai Optimalizálás:** Hiányzó és szöveges (`"Unknown"`) értékek kezelése, adattípusok konvertálása, megjelenési évek kinyerése, valamint a 109 millió soros `animelist.csv` memóriahatékony (downcastolt) beolvasása.
2. **A Nagy Háromszög (Minőség vs. Rajongás vs. Népszerűség):** A `Score`, `Favorites` és `Members` mutatók eloszlásának, rangkorrelációjának vizsgálata, új relatív mutatók (pl. *Kedvenc-arány*) bevezetése és az outlierek (kvadránsok) azonosítása.
3. **Háttértényezők szerepe (Típus, Kor, Epizódszám):** A folytatás-torzítás (*sequel bias*), az újdonság-torzítás (*recency bias*) és az elköteleződési hatás (*commitment effect*) statisztikai igazolása.
4. **Műfaj hatása:** Van-e olyan műfaj amit többen néznek, jobbra értékelnek vagy többen kedvencnek jelölnek mint másikokat?
5. **Mélyfúrás és Egyedi Mutatók:**
   * *Kaszálási és Halogatási Index:* Mely animéket hagyják félbe a legtöbben (`Drop Rate`), és melyek ragadnak be a „megnézendő” (`Plan to Watch`) listákba?
   * *Polarizációs Index:* A legmegosztóbb animék felkutatása a szavazatok szórása (`Score-1`...`Score-10`) alapján.
<!-- 5. **Big Data Insightok (`animelist.csv`):** A híres *„3 epizódos szabály”* empirikus tesztelése a lemorzsolódási adatokon, valamint a *„veterán kritikus-effektus”* vizsgálata (szigorúbban pontoznak-e azok, akik több száz animét láttak?). -->

---
*Az alábbi blokkban a szükséges Python könyvtárak importálásával és a globális megjelenítési beállításokkal indítjuk a munkát.*

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns


df_anime = pd.read_csv('data/anime.csv')
#df_animelist = pd.read_csv('data/animelist.csv')


### Segédfüggvények


In [ ]:
# Generated by Github Copilot
def duration_to_minutes(value):
    if pd.isna(value) or value == "Unknown":
        return pd.NA

    value = re.sub(r"\s+", " ", str(value)).strip()

    match = re.fullmatch(
        r"(?:(\d+(?:\.\d+)?)\s*hr\.\s*)?"
        r"(?:(\d+(?:\.\d+)?)\s*min\.\s*)?"
        r"(?:(\d+(?:\.\d+)?)\s*sec\.\s*)?"
        r"(?:per ep\.)?",
        value,
    )

    if match is None:
        raise ValueError(f"Unexpected duration format: {value!r}")

    hours, minutes, seconds = match.groups()

    return (
        float(hours or 0) * 60
        + float(minutes or 0)
        + float(seconds or 0) / 60
    )

# Generated by Github Copilot
def premiered_to_year(value):
    if pd.isna(value) or value == "Unknown":
        return pd.NA

    match = re.search(r"\d{4}", str(value).strip())


    if match is None:
        raise ValueError(f"Unexpected premiered format: {value!r}")

    return int(match.group(0))

### Adattisztítás és típuskonverzió (`anime.csv`)

A nyers adathalmazban a hiányzó értékeket szöveges `"Unknown"` karakterlánc jelöli, ami miatt a Pandas számos számszerű oszlopot szövegként (`object` típusként) olvas be. Az alábbi kódrészlet elvégzi az adatok szabványosítását és a későbbi elemzéshez szükséges új változók (feature-ök) kinyerését:

* **Hiányzó értékek és numerikus mezők kezelése:** Az `"Unknown"` értékeket szabványos `pd.NA`-ra cseréljük, majd az értékeléseket (`Score`, `Score-1`...`Score-10`), a rangsort (`Ranked`) és az epizódszámot (`Episodes`) numerikus típusúvá alakítjuk.
* **Időtartam és premier éve:** Segédfüggvények segítségével a szöveges `Duration` oszlopból kinyerjük az epizódok percekben mért hosszát (`Duration_minutes`), a szezonális `Premiered` mezőből pedig a megjelenés évét (`Premiered_year`).
* **Vetítési időszak feldolgozása:** Az `Aired` oszlop intervallumait (`" to "` mentén) szétválasztjuk kezdő (`start_date`) és befejező (`end_date`) dátumokra. Az egynapos megjelenéseknél (pl. mozifilmek, OVA-k) a kezdő és záró dátum megegyezik. A két időpont különbségéből kiszámítjuk a vetítés teljes időtartamát napokban (`length_days`).
* **Ellenőrzés:** A blokk végén egy összesítő táblázatban ellenőrizzük az oszloponkénti hiányzó értékek (`NaN`) számát és a véglegesített adattípusokat.

In [ ]:
#Data uses "Unknown" instead of NaN, so we need to replace it with NaN
df_anime.replace("Unknown", pd.NA, inplace=True)
string_numerics = ["Score", "Episodes", "Ranked", "Score-1", "Score-2", "Score-3", "Score-4", "Score-5", "Score-6", "Score-7", "Score-8", "Score-9", "Score-10"]
for col in string_numerics:
    df_anime[col] = pd.to_numeric(df_anime[col], errors='raise')

df_anime["Duration_minutes"] = (
    df_anime["Duration"]
    .apply(duration_to_minutes)
    .astype("Float64")
)

df_anime["Premiered_year"] = (
    df_anime["Premiered"]
    .apply(premiered_to_year)
    .astype("Int64")
)

#df_anime[["Start_date", "End_date", "Length_days"]] = df_anime["Aired"].apply(parse_air_date)

aired = (
    df_anime["Aired"]
    .astype("string")
    .str.strip()
    .replace(["Unknown", "?"], pd.NA)
)

dates = aired.str.split(" to ", n=1, expand=True)

# A single date is both the start and end date
start_raw = dates[0]
end_raw = dates[1].fillna(dates[0])

# If either side is "?", it becomes NaT
start_date = pd.to_datetime(start_raw, errors="coerce", format="mixed")
end_date = pd.to_datetime(end_raw, errors="coerce", format="mixed")

df_anime["Start_date"] = start_date
df_anime["End_date"] = end_date

# Automatically becomes NaN when either date is missing
df_anime["Length_days"] = (end_date - start_date).dt.days

# Relatív rajongási mutató: a nézők hány százaléka jelölte kedvencnek (%)
df_anime["Favorite_Percentage"] = (df_anime["Favorites"] / df_anime["Members"]) * 100

#df_anime["Actual_Members"] = df_anime["Watching"] + df_anime["Completed"] + df_anime["On-Hold"] + df_anime["Dropped"]

display(pd.DataFrame({
    "NaN count": df_anime.isna().sum(),
    "type": df_anime.dtypes
}))

In [ ]:
numerics = string_numerics + ["Popularity", "Members", "Favorites", "Watching", "Completed", "On-Hold", "Dropped", "Length_days", "Duration_minutes", "Premiered_year", "Favorite_Percentage"]

s_nancounts = df_anime[numerics].isna().sum()

desc = df_anime[numerics].describe().T

desc["NaNCount"] = s_nancounts

desc

# Korreláció

In [ ]:
#ai-usage: prompt1

# Csak az értékeléssel rendelkező és legalább 1000 fő által listázott animéket tartjuk meg
df_corr = df_anime.dropna(subset=['Score', 'Members', 'Favorites', 'Favorite_Percentage']).copy()
df_corr = df_corr[df_corr['Members'] >= 1000].copy()


# Elemzendő oszlopok és magyar címkéik a grafikonokhoz
cols = ['Score', 'Members', 'Favorites', 'Favorite_Percentage']
labels_heatmap = [
    'Átlagpontszám\n(Score)', 
    'Nézői elérés\n(Members)', 
    'Kedvencek száma\n(Favorites)', 
    'Rajongási arány\n(Favorites %)'
]

# Spearman-rangkorrelációs mátrix kiszámítása
spearman_matrix = df_corr[cols].corr(method='spearman')


fig, ax = plt.subplots(figsize=(8, 6), dpi=120)

# Felső háromszög maszkolása az áttekinthetőségért (opcionális, itt teljes mátrixot mutatunk)
sns.heatmap(
    spearman_matrix,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    square=True,
    linewidths=1.5,
    cbar_kws={"label": "Spearman-féle rangkorreláció (ρ)"},
    xticklabels=labels_heatmap,
    yticklabels=labels_heatmap,
    annot_kws={"size": 12},
    ax=ax
)

ax.set_title(
    "A Nagy Háromszög: Minőség, Elérés és Rajongás Korrelációja\n(Spearman ρ, szűrve: Members ≥ 1000)", 
    fontsize=13, fontweight="bold", pad=15
)
plt.xticks(rotation=0, fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.show()


# A vizualizációhoz log10 transzformációt alkalmazunk a ferde eloszlású változókra,
# hogy a scatter plotokon láthatóvá váljon a valódi szerkezet.
df_plot = pd.DataFrame()
df_plot['Score (1-10)'] = df_corr['Score']
df_plot['log10(Members)'] = np.log10(df_corr['Members'])
# Hozzáadunk 1-et a Favorites oszlophoz, hogy elkerüljük a log10(0) problémát
df_plot['log10(Favorites) + 1'] = np.log10(df_corr['Favorites'] + 1)
df_plot['log10(Favorite_Percentage) + 0.01'] = np.log10(df_corr['Favorite_Percentage'] + 0.01)


# PairGrid inicializálása
g = sns.PairGrid(df_plot, diag_sharey=False, corner=True)

# Alsó háromszög: Scatter plotok alacsony átlátszósággal (alpha) a sűrűség érzékeltetésére
g.map_lower(
    sns.scatterplot, 
    alpha=0.15, 
    s=15, 
    color="#1f77b4", 
    edgecolor="none"
)

# Átló: Hisztogramok és sűrűségfüggvények (KDE) az egyes változók eloszlásáról
g.map_diag(
    sns.histplot, 
    kde=True, 
    color="#2b5c8f", 
    bins=60, 
    edgecolor="white",
)

# Főcím beállítása
g.fig.subplots_adjust(top=0.92)
g.fig.suptitle(
    "Tulajdonságok Páronkénti Kapcsolata és Eloszlása (Scatter Plot Mátrix, Members ≥ 1000)", 
    fontsize=15, fontweight="bold"
)

plt.show()

### 1.1. A korrelációs hőtérkép és a páronkénti eloszlások (Pairplot) elemzése

A statisztikailag releváns mezőnyre (`Members >= 1000`) elvégzett Spearman-féle rangkorrelációs elemzés és a logaritmizált páronkénti pontfelhő-mátrix az alábbi főbb összefüggéseket tárja fel a minőség (`Score`), az elérés (`Members`) és a rajongás (`Favorites`, `Favorites %`) között:

#### 1. Korrelációs mintázatok (Spearman)
* **Elérés és nyers rajongás erős összefonódása:** A legerősebb kapcsolat a `Members` és a `Favorites` között mérhető. Ez egy természetes skálázódás hatás: minél több felhasználó vesz fel egy animét a listájára, annál nagyobb a potenciális bázis, amelyből a kedvenc-jelölések kikerülnek. A nyers `Favorites` szám tehát önmagában erősen népszerűség-vezérelt mutató.
* **Minőség vs. Nyers mutatók:** Az átlagpontszám (`Score`) közepesen erős, pozitív korrelációt mutat mind a nézőszámmal (`0.65`), mind a kedvencek számával (`0.68`). A jó minőségű animék tehát általánosságban szélesebb közönséget érnek el.
* **A relatív rajongási arány önállósodása:** Amikor a kedvencek számát megtisztítjuk a nézőszám hatásától (`Favorites % = Favorites / Members * 100`), a `Members`-szel vett korreláció `0.40`-re, a `Score`-ral vett korreláció pedig `0.48`-ra mérséklődik. Ez igazolja, hogy a **kult-faktor (relatív rajongási intenzitás)** egy önálló dimenzió: nem feltétlenül a legnézettebb animék váltják ki a legmélyebb szubjektív elköteleződést.

#### 2. Egyváltozós eloszlások (A Pairplot főátlója)
* **`Score (1-10)`:** Megközelítőleg haranggörbe (normális) eloszlást követ, amelynek csúcsa a `6.5 - 7.2` tartományban található, enyhe bal oldali elnyúlással (a nagyon gyenge, 2–4 pontos címek felé).
* **`log10(Members)` és `log10(Favorites) + 1`:** Az eredeti változók extrém módon jobbra ferdültek, és még a 10-es alapú logaritmikus skálán is erősen aszimmetrikusak. A `log10(Members)` eloszlása `3.0`-nál (az 1000 fős szűrési küszöbnél) magasan kezdődik, majd a milliós nézettségű (`> 6.0`) kasszasikerek felé fokozatosan elvékonyodik.
* **`log10(Favorite_Percentage) + 0.01`:** A fő tömeg egy szabályos haranggörbét alkot `-0.7` körül, míg a `-2.0` értéknél (`log10(0 + 0.01)`) egy éles tüske látható, amely a több mint 1000 nézővel, de **0 darab** kedvenc-jelöléssel rendelkező animéket tömöríti.

#### 3. Nemlineáris kapcsolatok és adatszerkezeti sajátosságok (Alsó háromszög)
* **`Score` vs. `Members`, `Favorites`, `Favorites %`:** Az első oszlop scatter plotjain jól látható, hogy `Score = 7.0` alatt a pontfelhő lapos és alacsony értékek körül sűrűsödik, azonban `7.5` felett a nézettség és a rajongási arány meredeken, nemlineárisan kilő. Ugyanakkor a bal oldalon (`Score < 6.0`, de magas Y érték) megjelennek azok a különleges outlierek, amelyek alacsony átlagpontszámuk ellenére jelentős elérést vagy rajongótábort építettek ki.
* **`log10(Members)` vs. `log10(Favorites) + 1`:** A két változó kapcsolata a magasabb tartományokban (`log10(Members) > 4.5`) rendkívül szoros, lineáris csatornába rendeződik. A tölcsér alsó részén látható vízszintes vonalak a kis egész számokból (`0, 1, 2, 3` kedvenc-jelölés) adódó diszkrét ugrásokat szemléltetik.
* **`log10(Members)` vs. `log10(Favorite_Percentage)`:** A második oszlop legalsó ábrájának bal alsó sarkában párhuzamos, lefelé ívelő diagonális sávok láthatók. Ezek matematikai műtermékek: mivel a `Favorites %` képlete $\frac{\text{Favorites}}{\text{Members}} \times 100$, a `-2.0` szinten húzódó vízszintes vonal pedig a 0 kedvenccel rendelkező címeket jelöli.

# Toplista átfedések

In [ ]:
#Ai-usage: prompt2

df_top = df_corr.copy()

# Top 50 halmazok definiálása
top_n = 50

top_score = set(df_top.nlargest(top_n, "Score")["MAL_ID"])
top_members = set(df_top.nlargest(top_n, "Members")["MAL_ID"])
top_favs = set(df_top.nlargest(top_n, "Favorites")["MAL_ID"])
top_fav_ratio = set(df_top.nlargest(top_n, "Favorite_Percentage")["MAL_ID"])

top_sets = {
    "Top 50 Értékelés (Score)": top_score,
    "Top 50 Népszerűség (Members)": top_members,
    "Top 50 Rajongás (Favorites)": top_favs,
    "Top 50 Rajongási Arány (Fav/Mem)": top_fav_ratio,
}

# 1. Átfedési mátrix (Közös elemek száma és % aránya)
matrix_names = list(top_sets.keys())
overlap_counts = np.zeros((len(matrix_names), len(matrix_names)), dtype=int)

for i, name1 in enumerate(matrix_names):
    for j, name2 in enumerate(matrix_names):
        overlap_counts[i, j] = len(top_sets[name1].intersection(top_sets[name2]))

overlap_df = pd.DataFrame(
    overlap_counts, index=matrix_names, columns=matrix_names
)

# Vizualizáció: Heatmap a PPT-hez
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    overlap_df,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar_kws={"label": "Közös animék száma (max 50)"},
    linewidths=1,
    annot_kws={"size": 13, "weight": "bold"},
    ax=ax,
)

ax.set_title(
    f"Top {top_n} Toplisták Páronkénti Átfedése (Közös címek száma)", pad=15
)
plt.xticks(rotation=20, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# 2. A "Szent Grál" animék: Amelyek mindhárom alaplistán (Score, Members, Favorites) Top 50-esek
all_three_core = top_score & top_members & top_favs
df_elit = df_top[df_top["MAL_ID"].isin(all_three_core)][
    [
        "Name",
        "Type",
        "Premiered_year",
        "Start_date",
        "Score",
        "Members",
        "Favorites",
        "Favorite_Percentage",
    ]
].sort_values(by="Score", ascending=False)

print(
    f"\nMindhárom alapmutatóban (Score, Members, Favorites) Top {top_n} anime ({len(df_elit)} db):"
)
display(df_elit.reset_index(drop=True))

### Elemzési Észrevételek: Top 50 Listák Átfedése és a „Elit” Csoport

Az aggregált mérőszámok összehasonlítása egyértelműen rámutat a minőség (`Score`), az elérés (`Members`) és az érzelmi elköteleződés (`Favorites` / arány) közötti dimenziók strukturális eltéréseire.

---

#### 1. Páronkénti Átfedések Vizsgálata (Heatmap Elemzés)

* **Erős kapcsolódás a volumenalapú metrikák között (Members vs. Favorites – 32 / 50 közös cím):**
  A legmagasabb átfedés a népszerűség és az abszolút kedvencek listája között figyelhető meg. Ez nem meglepő, mivel egy nagyobb felhasználói bázisból könnyebben gyűlik össze magas abszolút kedvenc-szám, még visszafogottabb konverziós arány mellett is.
* **Kritikai elismerés vs. Tömeges elérés (Score vs. Members – mindössze 10 / 50 közös cím):**
  A legkisebb átfedést a pontszám és a nézőszám mutatja. A legjobbra értékelt 50 cím 80%-a nem tartozik a legnézettebb 50 anime közé. Ez a jelenség a rétegműfajok magas értékeléséből, valamint a későbbi évadok túlélési torzításából (sequel bias) fakad, ahol a lemorzsolódott casual nézők már nem húzzák le az átlagot.
* **A rajongási arány függetlenítő szerepe (Fav/Mem arány):**
  A relatív rajongási arány (`Favorites / Members`) mind az értékeléssel (19 cím), mind a kedvencek abszolút számával (25 cím) kiegyensúlyozottabb kapcsolatban áll, mint a nyers eléréssel (16 cím). Ez megerősíti, hogy a normalizált arány képes kiszűrni a pusztán a nagy nézőszámból fakadó torzításokat.

---

#### 2. A Konszenzusos Elit („Elit” – Mindhárom Alapmutatóban Top 50)

A 17 562 animét számláló teljes adatbázisból mindössze **9 cím** képes egyszerre a legmagasabbra értékelt, a legnézettebb és a legtöbbször kedvencnek jelölt 50 anime közé kerülni:

* **Tökéletes konszenzus és elköteleződés:**
  A lista élén álló trió (*Fullmetal Alchemist: Brotherhood*, *Steins;Gate*, *Hunter x Hunter (2011)*) nem csupán a kritikai ranglétrát vezeti 9.10 feletti pontszámmal, de a nézőbázisuk több mint 8%-a kedvencnek is jelölte őket, ami a leghíresebb animék között is kiemelkedően magas konverziós arány.
* **Formátumbeli dinamika (Filmek jelenléte):**
  A listára két egész estés mozifilm (*Koe no Katachi*, *Kimi no Na wa.*) is felkerült. Bár ezek pontszáma eléri vagy megközelíti a 9.00-t, rajongási arányuk (~4.1–4.5%) elmarad a hosszabb sorozatokétól. Ez alátámasztja az elköteleződési hipotézist (commitment effect): a filmek rövidebb játékideje alatt nehezebb olyan mély kötődést kialakítani, mint egy több tíz epizódon átívelő TV-sorozat esetén.
* **Időtállóság és franchise-erő:**
  A *Code Geass* mindkét évadával képviselteti magát a szűk elitben, ami a 2000-es évek közepének ritka, fenntartható sikerességét bizonyítja. A *Cowboy Bebop* jelenléte pedig igazolja a nosztalgia-faktort: 1998-as bemutatója ellenére mindmáig megőrizte pozícióját mindhárom dimenzió élmezőnyében.

In [ ]:
#Ai-usage: prompt2

# Küszöbértékek dinamikus számítása kvantilisekkel a df_clean (Members >= 10,000) halmazon
q_mem_95 = df_top["Members"].quantile(0.95)
q_mem_90 = df_top["Members"].quantile(0.90)
q_mem_50 = df_top["Members"].quantile(0.50)

q_score_90 = df_top["Score"].quantile(0.90)
q_score_40 = df_top["Score"].quantile(0.40)

q_favratio_98 = df_top["Favorite_Percentage"].quantile(0.98)
q_favratio_30 = df_top["Favorite_Percentage"].quantile(0.30)


def classify_outliers(row):
    mem = row["Members"]
    score = row["Score"]
    fav_r = row["Favorite_Percentage"]

    # 1. Overhyped / Guilty Pleasure Blockbuster
    if mem >= q_mem_95 and score <= q_score_40:
        return "1. Overhyped (Magas nézettség, Alacsony értékelés)"

    # 2. Réteg-remekmű (Hidden Gem) - kizárjuk a még nem befejezett vagy nagyon kis mintás címeket
    elif score >= q_score_90 and mem <= 80000:
        return "2. Réteg-remekmű (Magas értékelés, Alacsony nézettség)"

    # 3. Kultikus Klasszikus (Die-hard Fanbase)
    elif fav_r >= q_favratio_98 and mem > q_mem_95:
        return "3. Kultikus Klasszikus (Magas rajongási arány, Magas nézettség)"

    # 4. Passzív Közönségsiker (Mindenki látta, kevesek kedvence)
    elif mem >= q_mem_90 and fav_r <= q_favratio_30:
        return "4. Passzív Közönségsiker (Magas nézettség, Alacsony rajongási arány)"
    
    else:
        return "Normál"


df_top["Outlier_Category"] = df_top.apply(classify_outliers, axis=1)

# Csoportok elemszámának és átlagos mutatóinak áttekintése
summary_outliers = (
    df_top.groupby("Outlier_Category")
    .agg(
        Darabszám=("MAL_ID", "count"),
        Átlag_Score=("Score", "mean"),
        Medián_Members=("Members", "median"),
        Átlag_Fav_Arány_Pct=("Favorite_Percentage", "mean"),
    )
    .round(2)
    .sort_values(by="Darabszám", ascending=True)
)

display(summary_outliers)

# Top 5 példa kiírása kategóriánként a PPT narratívához
for cat in sorted(df_top["Outlier_Category"].unique()):
    if cat == "Normál / Átlagos eloszlás":
        continue
    #print(f"\n--- {cat} (Top példák) ---")
    subset = df_top[df_top["Outlier_Category"] == cat]

    # Kategóriától függő rendezés a legjellemzőbb példákhoz
    if "1." in cat or "4." in cat:
        subset = subset.sort_values(by="Members", ascending=False)
    elif "2." in cat:
        subset = subset.sort_values(by="Score", ascending=False)
    else:
        subset = subset.sort_values(by="Favorite_Percentage", ascending=False)

    #display(subset[["Name","Type","Episodes","Premiered_year","Score","Members","Favorite_Percentage",]].head(6))

In [ ]:
#Ai-usage: prompt2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

ax1: plt.Axes = ax1
ax2: plt.Axes = ax2

# Színpaletta az outlier csoportoknak (a normál pontok halványszürkék a háttérben)
palette = {
    "Normál": "#d3d3d3",
    "1. Overhyped (Magas nézettség, Alacsony értékelés)": "#e74c3c",  # Piros
    "2. Réteg-remekmű (Magas értékelés, Alacsony nézettség)": "#2ecc71",  # Zöld
    "3. Kultikus Klasszikus (Magas rajongási arány, Magas nézettség)": "#9b59b6",  # Lila
    "4. Passzív Közönségsiker (Magas nézettség, Alacsony rajongási arány)": "#f39c12",  # Narancs
}

left_categories = [
    "1. Overhyped (Magas nézettség, Alacsony értékelés)",
    "2. Réteg-remekmű (Magas értékelés, Alacsony nézettség)",
]
right_categories = [
    "3. Kultikus Klasszikus (Magas rajongási arány, Magas nézettség)",
    "4. Passzív Közönségsiker (Magas nézettség, Alacsony rajongási arány)",
]

# Bal DF: minden, ami nem 1-es vagy 2-es kategória, "Normál" lesz
df_ax1 = df_top.copy()
df_ax1["Plot_Category"] = df_ax1["Outlier_Category"].where(
    df_ax1["Outlier_Category"].isin(left_categories), "Normál"
)
df_ax1 = df_ax1.sort_values(
    by="Plot_Category", key=lambda x: x != "Normál"
)

# Jobb DF: minden, ami nem 3-as vagy 4-es kategória, "Normál" lesz
df_ax2 = df_top.copy()
df_ax2["Plot_Category"] = df_ax2["Outlier_Category"].where(
    df_ax2["Outlier_Category"].isin(right_categories), "Normál"
)
df_ax2 = df_ax2.sort_values(
    by="Plot_Category", key=lambda x: x != "Normál"
)


# Rendezés, hogy a színes outlier pontok a szürke pontok FÖLÉ kerüljenek a rajzolásnál
df_plot = df_top.sort_values(
    by="Outlier_Category", key=lambda x: x != "Normál"
)

# Members (Log) vs. Score
sns.scatterplot(
    data=df_ax1,
    x="Members",
    y="Score",
    hue="Plot_Category",
    hue_order=["Normál"] + left_categories,
    palette=palette,
    alpha=0.75,
    s=45,
    ax=ax1,
    legend=False,
)
ax1.set_xscale("log")
ax1.set_title("Nézettség vs. Kritikai Értékelés")
ax1.set_xlabel("Nézői érdeklődés - Members (Logaritmikus skála)")
ax1.set_ylabel("Átlagos értékelés - Score")
#ax1.axhline(q_score_90, color="grey", linestyle="--", alpha=0.5)
#ax1.axvline(q_mem_95, color="grey", linestyle="--", alpha=0.5)
ax1.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f"{int(x):,}"))

# Favorite (Log) vs. Members (Log)
sns.scatterplot(
    data=df_ax2,
    x="Members",
    y="Favorite_Percentage",
    hue="Plot_Category",
    hue_order=["Normál"] + right_categories,
    palette=palette,
    alpha=0.75,
    s=45,
    ax=ax2,
)
ax2.set_title("Nézettség vs. Kedvencek száma")
ax2.set_ylabel("Rajongási arány - Favorite / Members (Logaritmikus skála)")
ax2.set_xlabel("Nézői érdeklődés - Members (Logaritmikus skála)")
ax2.set_xscale("log")
ax2.set_yscale("log")
#ax2.axhline(q_favratio_98, color="grey", linestyle="--", alpha=0.5)

# Jelmagyarázat elhelyezése az ábra alatt, prezentáció-barát módon
handles, labels = ax2.get_legend_handles_labels()
legend_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        label=label,
        markerfacecolor=color,
        markersize=8,
        alpha=0.85,
    )
    for label, color in palette.items()
]

ax2.get_legend().remove()
fig.legend(
    handles=legend_handles,
    #labels,
    loc="lower center",
    ncol=3,
    frameon=True,
    fontsize=10,
    bbox_to_anchor=(0.5, -0.08),

)

plt.suptitle(
    "A Nagy Háromszög Szétválása: Karakteres Anime Archetípusok (Members >= 10 000)",
    fontsize=16,
    weight="bold",
    y=1.02,
)
plt.tight_layout()
plt.show()

### Elemzési Észrevételek: Az Outlier Archetípusok Elhelyezkedése és Dinamikája

---

#### 1. Bal Panel: Nézettség vs. Átlagos Értékelés (`Members` vs. `Score`)

* **2. Réteg-remekművek (Zöld klaszter – Magas értékelés, Alacsony nézettség):**
  A bal felső kvadránsban koncentrálódó címek 10 000 és 80 000 közötti eléréssel érnek el kiemelkedő, 7.8–8.8 közötti pontszámokat. Ide túlnyomórészt olyan specifikus rétegműfajok vagy franchise-ok későbbi évadai tartoznak, amelyeket a casual közönség nem talál meg, viszont a szűk, elkötelezett célcsoport rendkívül magasra értékel.
* **1. Overhyped (Piros pontok – Magas nézettség, Alacsony értékelés):**
  A jobb alsó sarokban látható, 500 000 feletti elérésű, de 6.5 alatti pontszámmal bíró szűk csoport jól demonstrálja a mainstream expozíció árnyoldalát. Ezek a művek hatalmas kulturális vagy mém-hype-ot kaptak, így tömegek tekintették meg őket, ám a széles közönség szigorúan büntette a történetbeli vagy produkciós hiányosságokat.

---

#### 2. Jobb Panel: Nézettség vs. Rajongási Arány (`Members` vs. `Favorites %`)

* **3. Kultikus Klasszikusok (Lila klaszter – Magas rajongási arány, Magas nézettség):**
  A jobb felső szekcióban elhelyezkedő animék nemcsak milliós elérést értek el, hanem a nézőbázisuk kiemelkedően magas, 4–10% közötti hányada kedvencnek is jelölte őket. Ez a ritka csoport képviseli az iparág tartós oszlopait, amelyek képesek voltak a tömeges nézettséget mély, személyes kötődéssé konvertálni.
* **4. Passzív Közönségsikerek (Narancs pontok – Magas nézettség, Alacsony rajongási arány):**
  Közvetlenül a horizontális eloszlás alsó szélén (0.1% körüli kedvenc-aránnyal), több százezres nézettség mellett izolálódik ez a csoport. Bár sokan nézték meg őket, alig akad néző, aki a "kedvencek" listájára emelte őket. Ezek tipikusan könnyen fogyasztható, szezonális tartalmak, amelyek az adott időszakban felkapottak voltak, de maradandó érzelmi nyomot nem hagytak.

# Háttértényezők vizsgálata (Type, Kor, Episodes)

Ebben a fejezetben azt vizsgáljuk, hogy az animék strukturális jellemzői (formátum, megjelenési év, epizódszám) milyen torzításokat és mintázatokat okoznak az értékelésekben (`Score`), a népszerűségben (`Members`) és a rajongói elköteleződésben (`Favorite_Percentage`). A statisztikai zaj csökkentése érdekében az elemzést a legalább 1000 felhasználóval (`Members >= 1000`) rendelkező, érvényes pontszámú animékre korlátozzuk.

## Típus és Túlélési torzítás (Sequel / Movie Bias)

* **Módszertan:** A formátumok (`Type`) szerinti eloszlásokat Raincloud plottal (fél-hegedű sűrűségfüggvény + boxplot + szórásdiagram) ábrázoljuk a `Score` és a `Favorite_Percentage` (log10 skálán) mentén. A túlélési torzítás (Sequel Bias) kimutatásához szöveges mintázatok alapján szétválasztjuk az alapműveket és a folytatásokat, majd csoportosított oszlopdiagramon hasonlítjuk össze a medián mutatóikat.

In [ ]:
# import re
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import matplotlib.patches as mpatches
# import seaborn as sns

# # Prezentációs stílus beállítása
# plt.rcParams.update({
#     'figure.facecolor': 'white',
#     'axes.facecolor': '#F8F9FA',
#     'axes.edgecolor': '#CCCCCC',
#     'axes.grid': True,
#     'grid.alpha': 0.4,
#     'grid.color': '#CCCCCC',
#     'font.sans-serif': 'DejaVu Sans',
#     'axes.titlesize': 14,
#     'axes.titleweight': 'bold',
#     'axes.labelsize': 11,
#     'xtick.labelsize': 10,
#     'ytick.labelsize': 10
# })

# Globális szűrés a statisztikai zaj kizárására
df_filtered = df_anime[
    (df_anime['Members'] >= 1000) & 
    (df_anime['Score'].notna()) & 
    (df_anime['Type'].notna())
].copy()

# Favorite_Percentage biztosítása (ha még nincs, vagy 0 értékek kezelése log skálához)
if 'Favorite_Percentage' not in df_filtered.columns:
    df_filtered['Favorite_Percentage'] = (df_filtered['Favorites'] / df_filtered['Members']) * 100

df_filtered['Log10_Fav_Pct'] = np.log10(df_filtered['Favorite_Percentage'].replace(0, np.nan))

# ---------------------------------------------------------
# 1. ÁBRA: Kettős Raincloud Plot (Score és Favorite_Percentage)
# ---------------------------------------------------------
def draw_raincloud(ax, df, x_col, y_col, order, palette, xlabel, title):
    # 1. Fél-hegedű (Half-Violin / Cloud)
    sns.violinplot(
        data=df, x=x_col, y=y_col, order=order, palette=palette, hue=y_col, legend=False,
        inner=None, cut=0, alpha=0.6, ax=ax, density_norm='width'
    )
    # A hegedű alsó felének levágása, hogy valódi "felhő" legyen a felső részen
    for violin in ax.collections:
        bbox = violin.get_paths()[0].get_extents()
        y0 = (bbox.y0 + bbox.y1) / 2
        violin.set_clip_path(plt.Rectangle((bbox.x0, bbox.y0), bbox.width, y0 - bbox.y0, transform=ax.transData))

    # 2. Boxplot (Középső sáv)
    sns.boxplot(
        data=df, x=x_col, y=y_col, order=order, width=0.18,
        color='white', ax=ax, boxprops=dict(edgecolor='#333333', zorder=3),
        whiskerprops=dict(color='#333333', zorder=3),
        capprops=dict(color='#333333', zorder=3),
        medianprops=dict(color='#D62728', linewidth=2.5, zorder=4),
        showfliers=False
    )

    # 3. Jittered Scatter / Esőcseppek (Alulra tolva, mintavételezve a gyors és tiszta megjelenítésért)
    df_sample = df.groupby(y_col, group_keys=False).apply(
        lambda g: g.sample(min(len(g), 250), random_state=42)
    )
    df_sample[y_col] = df.loc[df_sample.index, y_col].to_numpy()
    for idx, cat in enumerate(order):
        sub = df_sample[df_sample[y_col] == cat][x_col].dropna()
        jitter = np.random.uniform(0.12, 0.35, size=len(sub))
        ax.scatter(
            sub, np.repeat(idx, len(sub)) + jitter,
            color=palette[idx], alpha=0.45, s=12, edgecolors='none', zorder=2
        )

    ax.set_title(title, pad=12)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('')

type_order = ['TV', 'Movie', 'OVA', 'ONA', 'Special']
df_types = df_filtered[df_filtered['Type'].isin(type_order)].copy()
palette_types = sns.color_palette("Set2", len(type_order))

fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)


draw_raincloud(
    axes[0], df_types, x_col='Score', y_col='Type',
    order=type_order, palette=palette_types,
    xlabel='Átlagos Értékelés (Score)',
    title='Értékelések (Score) eloszlása formátum szerint'
)

draw_raincloud(
    axes[1], df_types.dropna(subset=['Log10_Fav_Pct']), x_col='Log10_Fav_Pct', y_col='Type',
    order=type_order, palette=palette_types,
    xlabel='Rajongói Arány - Log10(Favorite %)',
    title='Rajongói elköteleződés (Favorite %) formátum szerint'
)

# Log10 tengely feliratainak visszaalakítása olvasható százalékra
xticks = [-2, -1, 0, 1]
axes[1].set_xticks(xticks)
axes[1].set_xticklabels(['0.01%', '0.1%', '1%', '10%'])

fig.suptitle('Formátumok összehasonlítása: Minőség vs. Rajongói bázis (Members >= 1000)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 2. ÁBRA: Túlélési torzítás (Sequel Bias) - Csoportosított oszlopdiagram
# ---------------------------------------------------------
sequel_pattern = re.compile(
    r'(?:2nd|3rd|4th|5th|6th|\bII\b|\bIII\b|\bIV\b|Season\s*[2-9]|Part\s*[2-9]|Final\s*Season|Sequel|Movie\s*[2-9]|:\s*Second)',
    re.IGNORECASE
)

df_tv_movie = df_filtered[df_filtered['Type'].isin(['TV', 'Movie'])].copy()
df_tv_movie['Sequel_Category'] = df_tv_movie['Name'].apply(
    lambda x: 'Folytatás / Későbbi rész' if sequel_pattern.search(str(x)) else 'Alapmű / 1. évad'
)

agg_sequel = df_tv_movie.groupby(['Type', 'Sequel_Category'], as_index=False).agg(
    Median_Score=('Score', 'median'),
    Median_Members=('Members', 'median'),
    Median_Fav_Pct=('Favorite_Percentage', 'median'),
    Count=('MAL_ID', 'count')
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
palette_seq = {'Alapmű / 1. évad': '#4C72B0', 'Folytatás / Későbbi rész': '#DD8452'}

metrics = [
    ('Median_Score', 'Medián Score (1-10)', 'Medián Értékelés (Score) ugrása', (6.0, 8.2)),
    ('Median_Members', 'Medián Members (fő)', 'Medián Nézőszám (Members) lemorzsolódása', None),
    ('Median_Fav_Pct', 'Medián Favorite (%)', 'Medián Rajongói Arány (Favorite %)', None)
]

for ax, (col, ylabel, title, ylim) in zip(axes, metrics):
    sns.barplot(
        data=agg_sequel, x='Type', y=col, hue='Sequel_Category',
        order=['TV', 'Movie'], palette=palette_seq, ax=ax, edgecolor='black'
    )
    ax.set_title(title, pad=10)
    ax.set_xlabel('Formátum (Type)')
    ax.set_ylabel(ylabel)
    if ylim:
        ax.set_ylim(ylim)
    
    # Értékek kiírása az oszlopok tetejére
    for container in ax.containers:
        fmt = '%.2f' if col != 'Median_Members' else '%.0f'
        ax.bar_label(container, fmt=fmt, padding=3, fontsize=10, fontweight='bold')
    
    if ax != axes[0]:
        ax.get_legend().remove()
    else:
        ax.legend(title='Kategória', loc='upper right')

fig.suptitle('Túlélési torzítás (Sequel Bias): Magasabb pontszám, de szűkebb közönség a folytatásoknál', fontsize=16, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

### Elemzési megállapítások: Formátumok és Túlélési torzítás (Sequel Bias)

**1. Formátumok összehasonlítása (Raincloud Plot)**
* **Minőség (Score):** A mozifilmek (Movie) és a TV-sorozatok (TV) medián értékelése a legmagasabb a formátumok közül. A filmek pontszám-eloszlása némileg feljebb tolódik, mint a többi kategóriáé.
* **Rajongói elköteleződés (Favorite %):** Bár a filmek magas pontszámokat kapnak, a valódi, mély rajongást a **TV-sorozatok** váltják ki. A logaritmikus tengelyen jól látható, hogy a TV-sorozatok medián rajongói aránya a legmagasabb, és az eloszlás felső sávja (a kiugró sikercímekkel) is itt a legszélesebb. Ez azzal magyarázható, hogy a több epizód több időt biztosít a nézőnek a karakterekhez és a világhoz való kötődés kialakítására.

**2. Túlélési torzítás a folytatásoknál (Oszlopdiagramok)**
* Az ábrák egyértelműen igazolják a túlélési torzítás (survivorship / sequel bias) jelenségét az anime-értékeléseknél.
* A TV-sorozatok és a filmek esetében is igaz, hogy **a folytatások (2.+ évadok, sequel filmek) szignifikánsan magasabb medián értékelést (TV-nél 6.94 vs. 7.49) és magasabb rajongói arányt érnek el**, mint az alapművek.
* **Oka:** Ez az önkiválasztási torzítás eredménye. Egy 2. vagy 3. évadot már jórészt csak az a célközönség kezd el nézni és pontozni, akiknek az első évad is tetszett. Mivel a csalódott nézők már az első évadnál lemorzsolódtak, a folytatások "hozott anyagból" dolgoznak, így a közönségük sokkal elkötelezettebb, ami mesterségesen felfelé hajtja a mutatókat az önálló, új címekhez képest.

## 2.2. Kor (`Premiered_year`) – Újdonság-torzítás (Recency Bias) vs. Nosztalgia-faktor

* **Módszertan:** Az időbeli trendeket egy kéttengelyes idősoros szalagdiagramon (Dual-Axis Quantile Ribbon Plot) vizsgáljuk 1980 és 2020 között. A premier évét a `Premiered_year` oszlopból, hiányzó érték esetén a `Start_date` évéből nyerjük ki. A bal oldali tengelyen a medián `Score` és `Favorite_Percentage` alakulását ábrázoljuk az interkvartilis terjedelemmel (25–75% kvantilis szalaggal), míg a jobb oldali tengelyen a nézői bázis (`Members`) robbanásszerű növekedését mutatjuk be.

In [ ]:
# Év kinyerése: Premiered_year használata, vagy fallback a Start_date évére
df_time = df_filtered.copy()
if 'Start_date' in df_time.columns:
    start_year = pd.to_datetime(df_time['Start_date'], errors='coerce').dt.year
    df_time['Year'] = df_time['Premiered_year'].fillna(start_year)
else:
    df_time['Year'] = df_time['Premiered_year']

df_time = df_time[(df_time['Year'] >= 1980) & (df_time['Year'] <= 2020)].copy()
df_time['Year'] = df_time['Year'].astype(int)

# Éves aggregációk és kvantilisek kiszámítása
yearly_stats = df_time.groupby('Year').agg(
    Score_Median=('Score', 'median'),
    Score_Q25=('Score', lambda x: np.percentile(x.dropna(), 25)),
    Score_Q75=('Score', lambda x: np.percentile(x.dropna(), 75)),
    Fav_Median=('Favorite_Percentage', 'median'),
    Fav_Q25=('Favorite_Percentage', lambda x: np.percentile(x.dropna(), 25)),
    Fav_Q75=('Favorite_Percentage', lambda x: np.percentile(x.dropna(), 75)),
    Members_Median=('Members', 'median'),
    Members_Total=('Members', 'sum'),
    Anime_Count=('MAL_ID', 'count')
).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(15, 10), sharex=True)

# ---------------------------------------------------------
# 1. PANEL: Score szalagdiagram + Összesített Members (Recency Bias)
# ---------------------------------------------------------
ax1 = axes[0]
ax1_twin = ax1.twinx()
ax1_twin.grid(False)

# Háttér oszlopok: Kumulált nézői aktivitás (Members_Total millióban)
ax1_twin.bar(
    yearly_stats['Year'], yearly_stats['Members_Total'] / 1e6,
    color='#B0C4DE', alpha=0.45, width=0.7, label='Összesített nézőszám (Millió Members)'
)
ax1_twin.set_ylabel('Összesített Members (Millió fő)', color='#3B5998', fontweight='bold')
ax1_twin.tick_params(axis='y', labelcolor='#3B5998')

# Előtér: Score medián és IQR szalag
ax1.plot(
    yearly_stats['Year'], yearly_stats['Score_Median'],
    color='#1F77B4', linewidth=2.5, marker='o', markersize=4, label='Medián Score'
)
ax1.fill_between(
    yearly_stats['Year'], yearly_stats['Score_Q25'], yearly_stats['Score_Q75'],
    color='#1F77B4', alpha=0.2, label='Score Interkvartilis sáv (25–75%)'
)
ax1.set_ylabel('Értékelés (Score)', color='#1F77B4', fontweight='bold')
ax1.tick_params(axis='y', labelcolor='#1F77B4')
ax1.set_title('Értékelések (Score) stabilitása és a 2012 utáni nézőszám-robbanás (Recency Bias)')

# Kombinált jelmagyarázat az 1. panelhez
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_twin.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

# ---------------------------------------------------------
# 2. PANEL: Favorite_Percentage szalagdiagram + Medián Members
# ---------------------------------------------------------
ax2 = axes[1]
ax2_twin = ax2.twinx()
ax2_twin.grid(False)

# Háttér oszlopok: Egy animére jutó medián Members (ezer fő)
ax2_twin.bar(
    yearly_stats['Year'], yearly_stats['Members_Median'] / 1e3,
    color='#C5B0D5', alpha=0.45, width=0.7, label='Medián Members / anime (Ezer fő)'
)
ax2_twin.set_ylabel('Medián Members / anime (Ezer fő)', color='#5D3A9B', fontweight='bold')
ax2_twin.tick_params(axis='y', labelcolor='#5D3A9B')

# Előtér: Favorite_Percentage medián és IQR szalag
ax2.plot(
    yearly_stats['Year'], yearly_stats['Fav_Median'],
    color='#D62728', linewidth=2.5, marker='s', markersize=4, label='Medián Favorite (%)'
)
ax2.fill_between(
    yearly_stats['Year'], yearly_stats['Fav_Q25'], yearly_stats['Fav_Q75'],
    color='#D62728', alpha=0.2, label='Favorite % Interkvartilis sáv (25-75%)'
)
ax2.set_ylabel('Rajongói Arány (Favorite %)', color='#D62728', fontweight='bold')
ax2.tick_params(axis='y', labelcolor='#D62728')
ax2.set_xlabel('Premier Éve (1980-2020)')
ax2.set_title('Nosztalgia-faktor vs. Modern tömegfogyasztás: Rajongói arány (Favorite %) és medián elérés')

# Kombinált jelmagyarázat a 2. panelhez
lines3, labels3 = ax2.get_legend_handles_labels()
lines4, labels4 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines3 + lines4, labels3 + labels4, loc='upper left')

plt.xticks(np.arange(1980, 2021, 5))
fig.suptitle('Időbeli dinamikák: Nosztalgia-faktor és Újdonság-torzítás (1980-2020)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Elemzési megállapítások: Időbeli dinamikák (1980–2020)

**1. Nézőszám-robbanás és Újdonság-torzítás (Recency Bias)**
* Az ábrák hátterében lévő oszlopdiagramok látványosan szemléltetik a MyAnimeList felhasználóbázisának (és a globális anime-fogyasztásnak) a robbanását. 2010 – és különösen 2012 – után mind az évente megjelenő animék összesített nézőszáma (felső panel), mind az egy animére jutó medián nézőszám (alsó panel) drasztikusan kilőtt.
* Emiatt a népszerűségi toplistákat (`Members` alapján) a 2010 utáni címek dominálják, ami komoly újdonság-torzítást (Recency Bias) jelent a kortárs sorozatok javára.

**2. Nosztalgia-faktor és Túlélési torzítás (Survivorship Bias)**
* Az alsó panelen a piros vonal (Medián Favorite %) és az azt körülvevő sáv (interkvartilis terjedelem) mutatja, hogy a 80-as és 90-es évek animéinél a rajongói arány szórása és felső kvartilise extrém magas.
* Ennek magyarázata a szelekciós emlékezet: a korábbi évtizedekből szinte csak a kultikus klasszikusok élték túl az idő próbáját. Akik ma egy 1985-ös vagy 1998-as animét megnéznek és értékelnek, azok jellemzően elkötelezett rajongók, ami mesterségesen magasan tartja a nosztalgikus címek `Favorite_Percentage` mutatóját.

**3. Tömegfogyasztás vs. Elköteleződés**
* A felső panelen látható, hogy a medián értékelés (`Score`) az elmúlt évtizedekben relatíve stabil maradt (bár a 2010-es években enyhén emelkedő trendet mutat). 
* Ugyanakkor az alsó panel rávilágít egy fontos kontrasztra: hiába nézik ma sokkal többen az animéket (magas `Members`), a modern, ipari léptékű szezonális tartalomfogyasztás miatt a medián rajongói arány lecsökkent és stabilizálódott egy alacsonyabb szinten. A kazuális nézők sokkal ritkábban emelnek be egy modern tömegszériát a "kedvencek" szűk elitjébe.

## 2.3. Epizódszám (`Episodes`) – Elköteleződési hatás (Commitment Effect)

In [ ]:
# Adatok előkészítése és iparági epizód-kategóriák (Cour bontás) létrehozása
df_ep = df_filtered[
    (df_filtered['Episodes'].notna()) & 
    (df_filtered['Episodes'] > 0) & 
    (df_filtered['Favorite_Percentage'] > 0)
].copy()

df_ep['Log10_Fav_Pct'] = np.log10(df_ep['Favorite_Percentage'])

bins = [0, 1, 10, 14, 21, 26, 100, np.inf]
labels = [
    '1 ep\n(Film/Spec)',
    '2–10 ep\n(Rövid)',
    '11–14 ep\n(1 Cour)',
    '15–21 ep\n(Köztes)',
    '22–26 ep\n(2 Cour)',
    '27–100 ep\n(Hosszú)',
    '100+ ep\n(Maratoni)'
]
df_ep['Ep_Category'] = pd.cut(df_ep['Episodes'], bins=bins, labels=labels)

# Kétpaneles, letisztult Boxplot ábra (2 változó / panel)
fig, axes = plt.subplots(1, 2, figsize=(18, 6.5), sharex=True)
palette_ep = sns.color_palette("Blues", len(labels))

# ---------------------------------------------------------
# 1. PANEL: Epizódszám kategória vs. Rajongói Arány (Favorite %)
# ---------------------------------------------------------
ax_fav = axes[0]
sns.boxplot(
    data=df_ep, x='Ep_Category', y='Log10_Fav_Pct',
    palette=palette_ep, ax=ax_fav, showfliers=False, width=0.5,
    boxprops=dict(edgecolor='#333333'),
    medianprops=dict(color='#D62728', linewidth=2.5),
)

medians_fav_log = df_ep.groupby('Ep_Category', observed=True)['Log10_Fav_Pct'].median()
medians_fav_raw = df_ep.groupby('Ep_Category', observed=True)['Favorite_Percentage'].median()

ax_fav.plot(
    range(len(labels)), medians_fav_log.values,
    color='#D62728', marker='o', linewidth=2, linestyle='--', label='Kategória Medián'
)

for idx, (log_val, raw_val) in enumerate(zip(medians_fav_log.values, medians_fav_raw.values)):
    ax_fav.text(
        idx, log_val + 0.08, f'{raw_val:.2f}%',
        ha='center', va='bottom', fontsize=9.5, fontweight='bold',
        color='#B22222', bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='none', alpha=0.85)
    )

fav_ticks = [0.01, 0.1, 1, 10]
ax_fav.set_yticks(np.log10(fav_ticks))
ax_fav.set_yticklabels([f'{x}%' for x in fav_ticks])
ax_fav.set_title('Rajongói elköteleződés (Favorite %) epizódszám szerint', pad=10)
ax_fav.set_xlabel('Epizódszám Kategóriák (Cour bontás)')
ax_fav.set_ylabel('Rajongói Arány - Favorite % (Log10 skála)')
ax_fav.legend(loc='upper left')

# ---------------------------------------------------------
# 2. PANEL: Epizódszám kategória vs. Értékelés (Score)
# ---------------------------------------------------------
ax_score = axes[1]
sns.boxplot(
    data=df_ep, x='Ep_Category', y='Score',
    palette=palette_ep,
    ax=ax_score, showfliers=False, width=0.5,
    boxprops=dict(edgecolor='#333333'),
    medianprops=dict(color='#D62728', linewidth=2.5),
)

medians_score = df_ep.groupby('Ep_Category', observed=True)['Score'].median()

ax_score.plot(
    range(len(labels)), medians_score.values,
    color='#D62728', marker='o', linewidth=2, linestyle='--', label='Kategória Medián'
)

for idx, val in enumerate(medians_score.values):
    ax_score.text(
        idx, val + 0.08, f'{val:.2f}',
        ha='center', va='bottom', fontsize=9.5, fontweight='bold',
        color='#B22222', bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='none', alpha=0.85)
    )

ax_score.set_title('Átlagos Értékelés (Score) epizódszám szerint', pad=10)
ax_score.set_xlabel('Epizódszám Kategóriák (Cour bontás)')
ax_score.set_ylabel('Értékelés (Score, 1–10 skála)')
ax_score.legend(loc='upper left')

fig.suptitle('Elköteleződési hatás (Commitment Effect): A sorozat hossza sokkal erősebben növeli a rajongást, mint a pontszámot', fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

### Elemzési megállapítások: Epizódszám és Elköteleződési hatás (Commitment Effect)

**1. A rajongás (Favorite %) drasztikus növekedése a hosszal**
* A bal oldali ábra egyértelműen bizonyítja az elköteleződési hatást: minél hosszabb egy sorozat, annál nagyobb eséllyel konvertálja a nézőit hardcore rajongóvá.
* Míg egy tipikus egy szezonos (1 cour, 11–14 részes) anime medián rajongói aránya mindössze **0,29%**, addig egy dupla hosszúságú (2 cour, 22–26 részes) sorozatnál ez a szám már **0,48%**-ra ugrik.
* A 100 epizód feletti (maratoni) szériák esetében az érték eléri a **0,90%**-ot. Ez azt mutatja, hogy a több képernyőidő és a karakterekkel töltött hosszú hónapok/évek sokkal mélyebb érzelmi kötődést építenek ki.

**2. A pontszámok (Score) mérsékeltebb reakciója**
* A jobb oldali ábra rávilágít, hogy a magasabb epizódszám nem húzza magával ugyanolyan mértékben az átlagos értékeléseket, mint a rajongást.
* Bár a leghosszabb (100+ részes) animék medián pontszáma a legmagasabb (**7,30**), a növekedés sokkal laposabb. Például egy 1 cour (6,88) és egy 2 cour (7,21) sorozat pontszámai között nincs akkora szakadék.
* **Konklúzió:** A nézők nem feltétlenül tartják objektíven "jobbnak" vagy hibátlanabbnak a hosszú sorozatokat (ezt mutatja a relatíve stabil Score), de a beléjük fektetett idő (sunk cost és kötődés) miatt sokkal nagyobb valószínűséggel emelik be őket a személyes "Kedvencek" (Favorites) listájukra.

In [ ]:
# Adatok előkészítése: Genres oszlop szétbontása (explode)
df_genres = df_filtered[df_filtered['Genres'].notna()].copy()
df_genres['Genre_List'] = df_genres['Genres'].str.split(', ')
df_exploded = df_genres.explode('Genre_List')

# Szűrés: Csak az érvényes és legalább 100 animével rendelkező műfajok (Top 20 gyakoriság szerint)
genre_counts = df_exploded['Genre_List'].value_counts()
top_genres = genre_counts[genre_counts >= 100].head(20).index

df_top_genres = df_exploded[df_exploded['Genre_List'].isin(top_genres)].copy()

# Műfajonkénti mediánok kiszámítása
genre_stats = df_top_genres.groupby('Genre_List', as_index=False).agg(
    Median_Score=('Score', 'median'),
    Median_Members=('Members', 'median'),
    Median_Fav_Pct=('Favorite_Percentage', 'median'),
    Anime_Count=('MAL_ID', 'count')
)

# Rendezés a Medián Score alapján (csökkenő sorrend a fentről lefelé olvasáshoz)
genre_stats = genre_stats.sort_values(by='Median_Score', ascending=False).reset_index(drop=True)
genre_order = genre_stats['Genre_List'].tolist()

# Globális mediánok a referenciavonalakhoz (az alap df_filtered táblából)
global_med_score = df_filtered['Score'].median()
global_med_members = df_filtered['Members'].median() / 1e3
global_med_fav = df_filtered['Favorite_Percentage'].median()

# ---------------------------------------------------------
# ÁBRA: Hárompaneles összehasonlítás a Top 20 műfaj mentén
# ---------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(20, 9), sharey=True)
palette_genres = sns.color_palette("tab20", len(genre_order))

# 1. PANEL: Medián Score
ax0 = axes[0]
sns.barplot(
    data=genre_stats, x='Median_Score', y='Genre_List',
    order=genre_order, palette=palette_genres, ax=ax0, edgecolor='#333333',
    hue="Genre_List", legend=False,
)
ax0.axvline(global_med_score, color='#D62728', linestyle='--', linewidth=2, label=f'Globális Medián ({global_med_score:.2f})')
ax0.set_xlim(6.0, genre_stats['Median_Score'].max() + 0.35)
ax0.set_title('Medián Értékelés (Score)', pad=10)
ax0.set_xlabel('Medián Score (1-10 skála)')
ax0.set_ylabel('Műfaj (Top 20 leggyakoribb)')
ax0.legend(loc='upper left')

for container in ax0.containers:
    ax0.bar_label(container, fmt='%.2f', padding=4, fontsize=9.5, fontweight='bold')

# 2. PANEL: Medián Members (Ezer főben)
ax1 = axes[1]
genre_stats['Median_Members_k'] = genre_stats['Median_Members'] / 1e3
sns.barplot(
    data=genre_stats, x='Median_Members_k', y='Genre_List',
    order=genre_order, palette=palette_genres, ax=ax1, edgecolor='#333333',
    hue="Genre_List", legend=False
)
ax1.axvline(global_med_members, color='#D62728', linestyle='--', linewidth=2, label=f'Globális Medián ({global_med_members:.1f}k)')
ax1.set_xlim(0, genre_stats['Median_Members_k'].max() * 1.18)
ax1.set_title('Medián Nézőszám (Members)', pad=10)
ax1.set_xlabel('Medián Members (Ezer fő)')
ax1.set_ylabel('')
ax1.legend(loc='upper left')

for container in ax1.containers:
    ax1.bar_label(container, fmt='%.1fk', padding=4, fontsize=9.5, fontweight='bold')

# 3. PANEL: Medián Favorite_Percentage (%)
ax2 = axes[2]
sns.barplot(
    data=genre_stats, x='Median_Fav_Pct', y='Genre_List',
    order=genre_order, palette=palette_genres, ax=ax2, edgecolor='#333333',
    hue="Genre_List", legend=False
)
ax2.axvline(global_med_fav, color='#D62728', linestyle='--', linewidth=2, label=f'Globális Medián ({global_med_fav:.2f}%)')
ax2.set_xlim(0, genre_stats['Median_Fav_Pct'].max() * 1.18)
ax2.set_title('Medián Rajongói Arány (Favorite %)', pad=10)
ax2.set_xlabel('Medián Favorite (%)')
ax2.set_ylabel('')
ax2.legend(loc='upper left')

for container in ax2.containers:
    ax2.bar_label(container, fmt='%.2f%%', padding=4, fontsize=9.5, fontweight='bold')

fig.suptitle('Műfajok hatása az anime statisztikáira: Kritikai elismerés vs. Tömeges elérés vs. Rajongás (Members >= 1000)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Műfajok szétbontása és a Top 20 leggyakoribb műfaj kiszűrése
df_genres = df_filtered[df_filtered['Genres'].notna()].copy()
df_genres['Genre_List'] = df_genres['Genres'].str.split(r',\s*')
df_exploded = df_genres.explode('Genre_List')

genre_counts = df_exploded['Genre_List'].value_counts()
top20_genres = genre_counts[genre_counts >= 100].head(20).index
df_top20 = df_exploded[df_exploded['Genre_List'].isin(top20_genres)].copy()

# Műfajonkénti mediánok kiszámítása
genre_matrix = df_top20.groupby('Genre_List', as_index=False).agg(
    Median_Score=('Score', 'median'),
    Median_Members_k=('Members', lambda x: np.median(x) / 1e3),
    Median_Fav_Pct=('Favorite_Percentage', 'median')
)

# Medián határértékek a kvadránsokhoz
x_mid = genre_matrix['Median_Members_k'].median()
y_score_mid = genre_matrix['Median_Score'].median()
y_fav_mid = genre_matrix['Median_Fav_Pct'].median()

# Diszkrét kategóriák képzése az 1. panelhez (Members vs. Score)
def assign_score_quadrant(row):
    if row['Median_Members_k'] >= x_mid and row['Median_Score'] >= y_score_mid:
        return 'Mainstream Sikerek (Magas elérés, Magas Score)'
    elif row['Median_Members_k'] < x_mid and row['Median_Score'] >= y_score_mid:
        return 'Kritikai / Réteg Kedvencek (Alacsony elérés, Magas Score)'
    elif row['Median_Members_k'] >= x_mid and row['Median_Score'] < y_score_mid:
        return 'Popcorn Műfajok (Magas elérés, Alacsony Score)'
    else:
        return 'Háttér Műfajok (Alacsony elérés, Alacsony Score)'

# Diszkrét kategóriák képzése a 2. panelhez (Members vs. Favorite %)
def assign_fav_quadrant(row):
    if row['Median_Members_k'] >= x_mid and row['Median_Fav_Pct'] >= y_fav_mid:
        return 'Blockbuster Kultusz (Magas elérés, Magas Favorite %)'
    elif row['Median_Members_k'] < x_mid and row['Median_Fav_Pct'] >= y_fav_mid:
        return 'Kult-klasszikusok (Alacsony elérés, Magas Favorite %)'
    elif row['Median_Members_k'] >= x_mid and row['Median_Fav_Pct'] < y_fav_mid:
        return 'Passzív Közönségsiker (Magas elérés, Alacsony Favorite %)'
    else:
        return 'Alacsony Kötődés (Alacsony elérés, Alacsony Favorite %)'

genre_matrix['Quad_Score'] = genre_matrix.apply(assign_score_quadrant, axis=1)
genre_matrix['Quad_Fav'] = genre_matrix.apply(assign_fav_quadrant, axis=1)

# Diszkrét színpaletták a 4-4 kvadránshoz
palette_score = {
    'Mainstream Sikerek (Magas elérés, Magas Score)': '#2E7D32',
    'Kritikai / Réteg Kedvencek (Alacsony elérés, Magas Score)': '#1565C0',
    'Popcorn Műfajok (Magas elérés, Alacsony Score)': '#EF6C00',
    'Háttér Műfajok (Alacsony elérés, Alacsony Score)': '#757575'
}

palette_fav = {
    'Blockbuster Kultusz (Magas elérés, Magas Favorite %)': '#2E7D32',
    'Kult-klasszikusok (Alacsony elérés, Magas Favorite %)': '#1565C0',
    'Passzív Közönségsiker (Magas elérés, Alacsony Favorite %)': '#EF6C00',
    'Alacsony Kötődés (Alacsony elérés, Alacsony Favorite %)': '#757575'
}

# ---------------------------------------------------------
# ÁBRA: Kétpaneles Pozicionálási Mátrix (Fix pontméret, diszkrét színek)
# ---------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(18, 7.5), sharex=True)

# 1. PANEL: Elérés (Members) vs. Minőség (Score)
ax1 = axes[0]
sns.scatterplot(
    data=genre_matrix, x='Median_Members_k', y='Median_Score',
    hue='Quad_Score', palette=palette_score,
    s=130, edgecolor='black', linewidth=1.2, alpha=0.9, ax=ax1, zorder=3
)

ax1.axvline(x_mid, color='#D62728', linestyle='--', linewidth=1.5, alpha=0.7, zorder=2)
ax1.axhline(y_score_mid, color='#D62728', linestyle='--', linewidth=1.5, alpha=0.7, zorder=2)

for _, row in genre_matrix.iterrows():
    ax1.annotate(
        row['Genre_List'],
        (row['Median_Members_k'], row['Median_Score']),
        xytext=(0, 7), textcoords='offset points',
        ha='center', va='bottom', fontsize=9, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='none', alpha=0.75)
    )

ax1.set_title('1. Elérés vs. Kritikai Minőség (Score)', pad=12)
ax1.set_xlabel('Medián Nézőszám - Members (Ezer fő)')
ax1.set_ylabel('Medián Értékelés (Score, 1–10 skála)')
ax1.margins(0.12)

# 2. PANEL: Elérés (Members) vs. Rajongás (Favorite %)
ax2 = axes[1]
sns.scatterplot(
    data=genre_matrix, x='Median_Members_k', y='Median_Fav_Pct',
    hue='Quad_Fav', palette=palette_fav,
    s=130, edgecolor='black', linewidth=1.2, alpha=0.9, ax=ax2, zorder=3
)

ax2.axvline(x_mid, color='#D62728', linestyle='--', linewidth=1.5, alpha=0.7, zorder=2)
ax2.axhline(y_fav_mid, color='#D62728', linestyle='--', linewidth=1.5, alpha=0.7, zorder=2)

for _, row in genre_matrix.iterrows():
    ax2.annotate(
        row['Genre_List'],
        (row['Median_Members_k'], row['Median_Fav_Pct']),
        xytext=(0, 7), textcoords='offset points',
        ha='center', va='bottom', fontsize=9, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='none', alpha=0.75)
    )

ax2.set_title('2. Elérés vs. Rajongói Elköteleződés (Favorite %)', pad=12)
ax2.set_xlabel('Medián Nézőszám - Members (Ezer fő)')
ax2.set_ylabel('Medián Rajongói Arány - Favorite (%)')
ax2.legend(loc='lower right', fontsize=8.5, title_fontsize=9.5)
ax2.margins(0.12)

# Jelmagyarázat elhelyezése az ábra alatt, prezentáció-barát módon
handles, labels = ax2.get_legend_handles_labels()
ax2.get_legend().remove()
ax1.get_legend().remove()
fig.legend(
    handles=handles,
    labels=labels,
    loc="lower center",
    ncol=3,
    frameon=True,
    fontsize=10,
    bbox_to_anchor=(0.5, -0.08),
    title='Pozicionálási Kvadránsok'
)

fig.suptitle('Műfaji Pozicionálási Mátrix: Nézői elérés, Minőség és Rajongás a Top 20 műfajban', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()